# mine-tuning-model

## 필수 라이브러리 설치

In [10]:
!pip install transformers datasets peft trl accelerate bitsandbytes -q
!pip install torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip freeze > /content/drive/MyDrive/Colab\ Notebooks/Github/mine-tuning-model/requirements-ft.txt

# 데이터 로드 및 Instruction 형식 변환

In [4]:
from datasets import load_dataset

# 데이터 로드
ds = load_dataset("naklecha/minecraft-question-answer-700k")
print(ds)

# Instruction 형식으로 변환
def format_instruction(example):
    return {
        "text": f"""<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
{example['question']}
<|im_end|>
<|im_start|>assistant
{example['answer']}
<|im_end|>"""
    }

train_data = ds["train"].map(format_instruction, remove_columns=["question", "answer", "source"])
print(f" 데이터 변환 완료: {len(train_data)}개")
print("\n샘플 확인:")
print(train_data[0]["text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/943 [00:00<?, ?B/s]

train.json:   0%|          | 0.00/314M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/694814 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'source'],
        num_rows: 694814
    })
})


Map:   0%|          | 0/694814 [00:00<?, ? examples/s]

 데이터 변환 완료: 694814개

샘플 확인:
<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
What is the first statistic to decrease when a player performs energy-intensive actions in Minecraft?
<|im_end|>
<|im_start|>assistant
Saturation is the first statistic to decrease when a player performs energy-intensive actions, and it must be completely depleted before the visible hunger meter begins decreasing.
<|im_end|>


# 모델, 토크나이저 로드

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen2.5-14B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"✅ 모델 로드 완료!")
print(f"디바이스: {model.device}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ 모델 로드 완료!
디바이스: cuda:0


# LoRa 어댑터 설정

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16, # LoRA 랭크 (높을수록 학습 파라미터 증가)
    lora_alpha=32, # 스케일링 계수
    target_modules=[
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported